In [ ]:
# Exercício 9 - Classificação Multiclasse (One-vs-Rest e One-vs-One)

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split
import seaborn as sns

# Dados
centers = [[-5, 0], [0, 1.5], [5, -1], [10, 1.5], [15, 0]]
x, y = make_blobs(n_samples=1000, centers=centers, random_state=42)

# Dados com bias
X = np.c_[np.ones((len(x), 1)), x]

# Funções auxiliares
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def error_logistic(x, a, y, epsilon=1e-7):
    h = sigmoid(x.dot(a))
    return -np.mean(y * np.log(h + epsilon) + (1 - y) * np.log(1 - h + epsilon))

def logistic_regression(x, y, alpha=0.1, n_iterations=40000):
    a = np.random.randn(x.shape[1], 1)
    for _ in range(n_iterations):
        h = sigmoid(x.dot(a))
        gradient = (1/len(y)) * x.T.dot(h - y)
        a = a - alpha * gradient
    return a

# One-vs-Rest
classes = np.unique(y)
models_ovr = {}

for c in classes:
    y_bin = (y == c).astype(int).reshape(-1,1)
    a = logistic_regression(X, y_bin)
    models_ovr[c] = a

# Predição One-vs-Rest
pred_scores = np.hstack([sigmoid(X.dot(models_ovr[c])) for c in classes])
y_pred_ovr = np.argmax(pred_scores, axis=1)

# Avaliação One-vs-Rest
mat = confusion_matrix(y, y_pred_ovr)
plt.figure(figsize=(6,6))
sns.heatmap(mat, annot=True, fmt='d', cmap="Blues")
plt.title('Matriz de Confusão - One-vs-Rest')
plt.xlabel('Verdadeiro')
plt.ylabel('Predito')
plt.show()

acc_ovr = accuracy_score(y, y_pred_ovr)
print(f'Acurácia One-vs-Rest: {acc_ovr:.4f}')

# One-vs-One
from itertools import combinations
pairs = list(combinations(classes, 2))
models_ovo = {}
votes = np.zeros((len(y), len(classes)))

for (i, j) in pairs:
    idx = np.where((y == i) | (y == j))[0]
    X_pair = X[idx]
    y_pair = y[idx]
    y_bin = (y_pair == i).astype(int).reshape(-1,1)
    a = logistic_regression(X_pair, y_bin)
    models_ovo[(i, j)] = a
    
    pred = sigmoid(X_pair.dot(a)) >= 0.5
    votes[idx, i] += (pred.ravel() == 1)
    votes[idx, j] += (pred.ravel() == 0)

y_pred_ovo = np.argmax(votes, axis=1)

# Avaliação One-vs-One
mat = confusion_matrix(y, y_pred_ovo)
plt.figure(figsize=(6,6))
sns.heatmap(mat, annot=True, fmt='d', cmap="Blues")
plt.title('Matriz de Confusão - One-vs-One')
plt.xlabel('Verdadeiro')
plt.ylabel('Predito')
plt.show()

acc_ovo = accuracy_score(y, y_pred_ovo)
print(f'Acurácia One-vs-One: {acc_ovo:.4f}')

# Comparação
if acc_ovo > acc_ovr:
    print('One-vs-One teve melhor desempenho.')
elif acc_ovo < acc_ovr:
    print('One-vs-Rest teve melhor desempenho.')
else:
    print('Ambos tiveram desempenho igual.')
